In [1]:
#CNN 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ===== Paths & setup =====
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/ALL"   
IMG_SIZE = (256, 256)  
BATCH_SIZE = 8   # lower if you run into OOM
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

# Split val/test
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Baseline CNN =====
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # Block 1
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    # Dense layers
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_baseline_cnn(IMG_SIZE + (3,), num_classes)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ===== Train =====
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on test set =====
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")




2025-09-13 18:00:18.802025: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-13 18:00:20.460887: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 20000 files belonging to 4 classes.
Using 17000 files for training.


2025-09-13 18:00:26.048292: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22280 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:63:00.0, compute capability: 8.6
2025-09-13 18:00:26.049934: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22280 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:64:00.0, compute capability: 8.6


Found 20000 files belonging to 4 classes.
Using 3000 files for validation.
Classes: ['all_benign', 'all_early', 'all_pre', 'all_pro']
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 256, 256, 3)       0         
                                                                 
 rescaling (Rescaling)       (None, 256, 256, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 256, 256, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 128, 128, 32)      0         
 D)                                                              
                                                           

2025-09-13 18:00:40.876409: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 375 of 1000
2025-09-13 18:00:50.864943: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 658 of 1000
2025-09-13 18:00:59.416707: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-13 18:01:01.386083: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-13 18:01:02.137753: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-09-13 18:01:02.333422: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f5fe44bfb20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-13 18:01:02.333483: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor devi

2125/2125 [==============================] - 76s 19ms/step - loss: 0.6824 - accuracy: 0.7141 - val_loss: 0.5550 - val_accuracy: 0.9024
Epoch 2/20
2125/2125 [==============================] - 22s 9ms/step - loss: 0.3232 - accuracy: 0.8845 - val_loss: 0.2088 - val_accuracy: 0.9184
Epoch 3/20
2125/2125 [==============================] - 22s 10ms/step - loss: 0.2361 - accuracy: 0.9165 - val_loss: 0.1730 - val_accuracy: 0.9398
Epoch 4/20
2125/2125 [==============================] - 22s 9ms/step - loss: 0.1940 - accuracy: 0.9318 - val_loss: 0.1470 - val_accuracy: 0.9606
Epoch 5/20
2125/2125 [==============================] - 22s 9ms/step - loss: 0.1610 - accuracy: 0.9423 - val_loss: 0.1311 - val_accuracy: 0.9566
Epoch 6/20
2125/2125 [==============================] - 22s 9ms/step - loss: 0.1364 - accuracy: 0.9516 - val_loss: 0.1257 - val_accuracy: 0.9572
Epoch 7/20
2125/2125 [==============================] - 22s 10ms/step - loss: 0.1206 - accuracy: 0.9572 - val_loss: 0.1730 - val_accuracy: 

In [1]:
# ResNet
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ===== Paths & setup =====
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/ALL"    
IMG_SIZE = (256, 256)  
BATCH_SIZE = 8   # adjust if OOM
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)

valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

# Split val/test
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Residual block =====
def residual_block(x, filters, downsample=False):
    shortcut = x

    # First conv
    x = layers.Conv2D(filters, 3, padding="same", strides=(2 if downsample else 1),
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second conv
    x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)

    # Match shortcut shape if downsampling
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=2, padding="same",
                                 kernel_initializer="he_normal")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Add skip connection
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

# ===== Baseline ResNet model =====
def build_resnet(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # Stem
    x = layers.Conv2D(32, 7, strides=2, padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    # Residual blocks
    x = residual_block(x, 32)
    x = residual_block(x, 32)
    x = residual_block(x, 64, downsample=True)
    x = residual_block(x, 64)
    x = residual_block(x, 128, downsample=True)
    x = residual_block(x, 128)

    # Classifier head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_resnet(IMG_SIZE + (3,), num_classes)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ===== Train =====
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate =====
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")


2025-09-13 22:15:52.647246: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-13 22:15:54.279608: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 20000 files belonging to 4 classes.
Using 17000 files for training.


2025-09-13 22:15:59.822816: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22280 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:63:00.0, compute capability: 8.6
2025-09-13 22:15:59.824514: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22280 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:64:00.0, compute capability: 8.6


Found 20000 files belonging to 4 classes.
Using 3000 files for validation.
Classes: ['all_benign', 'all_early', 'all_pre', 'all_pro']
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 sequential (Sequential)     (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 rescaling (Rescaling)       (None, 256, 256, 3)          0         ['sequential[0][0]']          
                                                                                                  
 conv2d (Conv2D)             (None, 128, 128, 32)         4

2025-09-13 22:16:19.250992: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 401 of 1000
2025-09-13 22:16:29.230401: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 820 of 1000
2025-09-13 22:16:33.080486: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-13 22:16:35.020314: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-13 22:16:36.313668: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-09-13 22:16:36.398471: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f920802d560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-13 22:16:36.398519: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor devi

2125/2125 [==============================] - 96s 25ms/step - loss: 0.2627 - accuracy: 0.9141 - val_loss: 4.7533 - val_accuracy: 0.4271
Epoch 2/20
2125/2125 [==============================] - 41s 18ms/step - loss: 0.1087 - accuracy: 0.9646 - val_loss: 0.8510 - val_accuracy: 0.6705
Epoch 3/20
2125/2125 [==============================] - 40s 18ms/step - loss: 0.0752 - accuracy: 0.9756 - val_loss: 1.6581 - val_accuracy: 0.6845
Epoch 4/20
2125/2125 [==============================] - 41s 18ms/step - loss: 0.0605 - accuracy: 0.9812 - val_loss: 7.2955 - val_accuracy: 0.5180
Epoch 5/20
2125/2125 [==============================] - 41s 18ms/step - loss: 0.0575 - accuracy: 0.9826 - val_loss: 1.3663 - val_accuracy: 0.6277
Epoch 6/20
2125/2125 [==============================] - 41s 19ms/step - loss: 0.0481 - accuracy: 0.9856 - val_loss: 0.9228 - val_accuracy: 0.7707
Epoch 7/20
2125/2125 [==============================] - 41s 18ms/step - loss: 0.0387 - accuracy: 0.9872 - val_loss: 0.0964 - val_accura

In [3]:
# VGGNet 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ===== Paths & setup =====
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/ALL"  
IMG_SIZE = (256, 256)  
BATCH_SIZE = 8        # lower if you hit OOM
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)
valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

# Split val/test
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== VGG-style block helper =====
def vgg_block(x, filters, convs=2):
    for _ in range(convs):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu",
                          kernel_initializer="he_normal")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    return x

# ===== Baseline VGG-style model (roughly VGG11/VGG13 sized) =====
def build_vgg(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # VGG-style feature extractor
    x = vgg_block(x, 32, convs=1)   # Block 1
    x = vgg_block(x, 64, convs=1)   # Block 2
    x = vgg_block(x, 128, convs=2)  # Block 3
    x = vgg_block(x, 256, convs=2)  # Block 4

    # Classifier head
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_vgg(IMG_SIZE + (3,), num_classes)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ===== Train =====
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate =====
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")


Found 20000 files belonging to 4 classes.
Using 17000 files for training.
Found 20000 files belonging to 4 classes.
Using 3000 files for validation.
Classes: ['all_benign', 'all_early', 'all_pre', 'all_pro']
Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 256, 256, 3)]     0         
                                                                 
 sequential_2 (Sequential)   (None, 256, 256, 3)       0         
                                                                 
 rescaling_2 (Rescaling)     (None, 256, 256, 3)       0         
                                                                 
 conv2d_21 (Conv2D)          (None, 256, 256, 32)      896       
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 128, 128, 32)      0         
 g2D)                                            

In [4]:
# GoogLeNet 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# ===== Paths & setup =====
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/ALL"
IMG_SIZE = (256, 256)  
BATCH_SIZE = 8        
SEED = 1337

data_dir = pathlib.Path(DATA_DIR)

# ===== Dataset split: train (70%), val (15%), test (15%) =====
train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)
valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

# Split val/test
val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# ===== Prefetch for performance =====
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# ===== Data augmentation =====
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# ===== Conv -> BN -> ReLU helper =====
def CBR(x, filters, k, s=1, p="same"):
    x = layers.Conv2D(filters, k, strides=s, padding=p,
                      use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

# ===== Inception (v1-style) block =====
def inception_block(x, f1, f3r, f3, f5r, f5, fpool):
    """
    Branches:
      - 1x1
      - 1x1 -> 3x3
      - 1x1 -> 5x5 (implemented as two 3x3 for efficiency)
      - 3x3 pool -> 1x1
    """
    b1 = CBR(x, f1, 1)

    b2 = CBR(x, f3r, 1)
    b2 = CBR(b2, f3, 3)

    b3 = CBR(x, f5r, 1)
    b3 = CBR(b3, f5, 3)
    b3 = CBR(b3, f5, 3)

    b4 = layers.MaxPooling2D(3, strides=1, padding="same")(x)
    b4 = CBR(b4, fpool, 1)

    return layers.Concatenate()([b1, b2, b3, b4])

# ===== Inception Reduction block (downsampling) =====
def inception_reduction(x, f3, f5r, f5):
    """
    Branches that reduce spatial dims:
      - 3x3 stride-2
      - 1x1 -> 3x3 -> 3x3 stride-2
      - 3x3 maxpool stride-2
    """
    b1 = CBR(x, f3, 3, s=2, p="same")

    b2 = CBR(x, f5r, 1)
    b2 = CBR(b2, f5, 3)
    b2 = CBR(b2, f5, 3, s=2)

    b3 = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    return layers.Concatenate()([b1, b2, b3])

# ===== Baseline Inception-style model =====
def build_inception_baseline(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    # Stem
    x = CBR(x, 32, 7, s=2)                # 512 -> 256
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)  # 256 -> 128
    x = CBR(x, 64, 3)

    # Inception stacks
    x = inception_block(x, f1=32, f3r=32, f3=48, f5r=8,  f5=16, fpool=16)
    x = inception_block(x, f1=48, f3r=40, f3=64, f5r=12, f5=24, fpool=24)

    # Reduction
    x = inception_reduction(x, f3=96, f5r=32, f5=48)  # 128 -> 64

    # More Inception blocks
    x = inception_block(x, f1=64, f3r=48, f3=96, f5r=16, f5=32, fpool=32)
    x = inception_block(x, f1=80, f3r=56, f3=112, f5r=20, f5=40, fpool=40)

    # Second reduction
    x = inception_reduction(x, f3=128, f5r=48, f5=80)  # 64 -> 32

    # Head
    x = inception_block(x, f1=96, f3r=64, f3=128, f5r=24, f5=48, fpool=48)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = build_inception_baseline(IMG_SIZE + (3,), num_classes)
model.compile(optimizer="adam",
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

model.summary()

# ===== Train =====
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate =====
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

Found 20000 files belonging to 4 classes.
Using 17000 files for training.
Found 20000 files belonging to 4 classes.
Using 3000 files for validation.
Classes: ['all_benign', 'all_early', 'all_pre', 'all_pro']
Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_4 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 sequential_3 (Sequential)   (None, 256, 256, 3)          0         ['input_4[0][0]']             
                                                                                                  
 rescaling_3 (Rescaling)     (None, 256, 256, 3)          0         ['sequential_3[0][0]']        
                                                                                  

In [1]:
# EfficientNet 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib

# =========================
# Paths & Dataset
# =========================
DATA_DIR = "/home/jovyan/.cache/kagglehub/datasets/obulisainaren/multi-cancer/versions/3/Multi Cancer/Multi Cancer/ALL"
IMG_SIZE = (256, 256)                # smaller than 512 to save memory
BATCH_SIZE = 8
SEED = 1337
data_dir = pathlib.Path(DATA_DIR)

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED
)
valtest_ds = keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED
)

val_batches = int(0.5 * tf.data.experimental.cardinality(valtest_ds).numpy())
val_ds = valtest_ds.take(val_batches)
test_ds = valtest_ds.skip(val_batches)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# =========================
# Building Blocks
# =========================
def SE(x, se_ratio=0.25):
    """Squeeze-and-Excitation."""
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D(keepdims=True)(x)
    se = layers.Conv2D(int(filters * se_ratio), 1, activation="swish")(se)
    se = layers.Conv2D(filters, 1, activation="sigmoid")(se)
    return layers.Multiply()([x, se])

def MBConv(x, out_ch, expand=4, kernel=3, stride=1, se_ratio=0.25, drop_rate=0.0):
    """Minimal EfficientNet-style MBConv block (no fused variant)."""
    in_ch = x.shape[-1]
    residual = x

    # Expansion
    mid_ch = int(in_ch * expand)
    if expand != 1:
        x = layers.Conv2D(mid_ch, 1, padding="same", use_bias=False,
                          kernel_initializer="he_normal")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("swish")(x)

    # Depthwise
    x = layers.DepthwiseConv2D(kernel, strides=stride, padding="same",
                               use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)

    # SE
    if se_ratio and se_ratio > 0:
        x = SE(x, se_ratio=se_ratio)

    # Projection
    x = layers.Conv2D(out_ch, 1, padding="same", use_bias=False,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)

    # Skip connection (if shape matches)
    if stride == 1 and in_ch == out_ch:
        if drop_rate and drop_rate > 0:
            x = layers.Dropout(drop_rate)(x)
        x = layers.Add()([x, residual])
    return x

def round_filters(filters, width_mult=1.0):
    return max(8, int(filters * width_mult))

def round_repeats(repeats, depth_mult=1.0):
    import math
    return max(1, int(math.ceil(repeats * depth_mult)))

# =========================
# Model (EfficientNet-like)
# =========================
def build_efficientnet_like(input_shape, num_classes, width_mult=1.0, depth_mult=1.0, drop_rate=0.2):
    inputs = keras.Input(shape=input_shape)

    # Data augmentation + basic rescale
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.Rescaling(1./255)(x)

    # Stem
    stem_ch = round_filters(32, width_mult)
    x = layers.Conv2D(stem_ch, 3, strides=2, padding="same", use_bias=False,
                      kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)

    # Stages: (out_channels, repeats, kernel, stride, expand)
    cfgs = [
        # out, r, k, s, t
        (16,  1, 3, 1, 1),   # MBConv1
        (24,  2, 3, 2, 4),   # MBConv6-ish (expand=4 to keep it light)
        (40,  2, 5, 2, 4),
        (80,  3, 3, 2, 4),
        (112, 3, 5, 1, 4),
        (192, 4, 5, 2, 4),
        (320, 1, 3, 1, 4),
    ]

    block_idx = 0
    total_blocks = sum(round_repeats(r, depth_mult) for (_, r, _, _, _) in cfgs)

    for out_ch, repeats, k, s, t in cfgs:
        out_ch = round_filters(out_ch, width_mult)
        reps = round_repeats(repeats, depth_mult)

        for i in range(reps):
            stride = s if i == 0 else 1
            # simple linear drop schedule
            b_drop = drop_rate * float(block_idx) / max(1, total_blocks - 1)
            x = MBConv(x, out_ch=out_ch, expand=t, kernel=k, stride=stride,
                       se_ratio=0.25, drop_rate=b_drop)
            block_idx += 1

    # Head
    head_ch = round_filters(1280, width_mult)
    x = layers.Conv2D(head_ch, 1, use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(drop_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

# Build the model (roughly around B0 scale)
model = build_efficientnet_like(IMG_SIZE + (3,), num_classes,
                                width_mult=1.0, depth_mult=1.0, drop_rate=0.2)

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# =========================
# Train & Evaluate
# =========================
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

2025-09-13 23:05:23.114847: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-13 23:05:24.831357: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Found 20000 files belonging to 4 classes.
Using 17000 files for training.


2025-09-13 23:05:30.313894: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22280 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:63:00.0, compute capability: 8.6
2025-09-13 23:05:30.315534: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22280 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:64:00.0, compute capability: 8.6


Found 20000 files belonging to 4 classes.
Using 3000 files for validation.
Classes: ['all_benign', 'all_early', 'all_pre', 'all_pro']
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 256, 256, 3)]        0         []                            
                                                                                                  
 random_flip (RandomFlip)    (None, 256, 256, 3)          0         ['input_1[0][0]']             
                                                                                                  
 random_rotation (RandomRot  (None, 256, 256, 3)          0         ['random_flip[0][0]']         
 ation)                                                                                           
                                                           

2025-09-13 23:05:54.763507: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-09-13 23:06:08.030605: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:422] Filling up shuffle buffer (this may take a while): 469 of 1000
2025-09-13 23:06:18.010184: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] Shuffle buffer filled.
2025-09-13 23:06:19.939007: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-09-13 23:06:20.905230: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-09-13 23:06:21.119944: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x560fb46fa7c0 initialized for platform CUDA (this does not guarantee that XLA will be use

2125/2125 [==============================] - 222s 68ms/step - loss: 0.2975 - accuracy: 0.8923 - val_loss: 1.3232 - val_accuracy: 0.8663 - lr: 3.0000e-04
Epoch 2/20
2125/2125 [==============================] - 139s 64ms/step - loss: 0.0906 - accuracy: 0.9711 - val_loss: 0.0803 - val_accuracy: 0.9779 - lr: 3.0000e-04
Epoch 3/20
2125/2125 [==============================] - 139s 64ms/step - loss: 0.0550 - accuracy: 0.9822 - val_loss: 0.3504 - val_accuracy: 0.8830 - lr: 3.0000e-04
Epoch 4/20
2125/2125 [==============================] - 139s 64ms/step - loss: 0.0450 - accuracy: 0.9870 - val_loss: 0.1322 - val_accuracy: 0.9472 - lr: 3.0000e-04
Epoch 5/20
2125/2125 [==============================] - 139s 65ms/step - loss: 0.0190 - accuracy: 0.9953 - val_loss: 0.0024 - val_accuracy: 1.0000 - lr: 1.5000e-04
Epoch 6/20
2125/2125 [==============================] - 139s 65ms/step - loss: 0.0136 - accuracy: 0.9958 - val_loss: 0.0650 - val_accuracy: 0.9880 - lr: 1.5000e-04
Epoch 7/20
2125/2125 [=====